In [4]:
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import random as rd
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, pipeline
import torch
from tqdm import tqdm
import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from peft import LoraConfig, get_peft_model, TaskType
from evaluate import load
from sklearn.model_selection import train_test_split

In [6]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [7]:
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base") #, device_map="cuda", torch_dtype=torch.bfloat16)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [8]:
ds_train = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[:80%]")
ds_test = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[80%:]")
rankings_train = np.load('/kaggle/input/xenc-scores-distilroberta/xenc_scores_train-stsb-distilroberta-base.npy')
rankings_test = np.load('/kaggle/input/xenc-scores-distilroberta/xenc_scores_test-stsb-distilroberta-base.npy')

k = 3

README.md:   0%|          | 0.00/2.64k [00:00<?, ?B/s]

rag_instruct.json:   0%|          | 0.00/296M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40541 [00:00<?, ? examples/s]

In [9]:
def build_df(ds, doc_rankings):
    docs = [d for s in ds['documents'] for d in s]
    questions = [q for q in ds['question']]
    answers = [a for a in ds['answer']]
    
    data = []

    for i, (q, a) in enumerate(zip(questions, answers)):
        ranked_indices = [int(t[1]) for t in doc_rankings[i][:k]]
        top_docs = [docs[i + idx] for idx in ranked_indices]
        data.append({
            'question': q,
            'answer': a,
            'topk_documents': top_docs
        })

    df = pd.DataFrame(data)
    topk_df = Dataset.from_pandas(df)
    return topk_df

In [10]:
topk_ds_train = build_df(ds_train, rankings_train)
topk_ds_testval = build_df(ds_test, rankings_test)

topk_ds_split = topk_ds_testval.train_test_split(
    test_size=0.25,
    shuffle=True,
    seed=42
)

topk_ds_val = topk_ds_split["train"]
topk_ds_test = topk_ds_split["test"] 

### Performance Evaluation

In [11]:
import re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces
    return text

def evaluate_model(model, tokenizer, test_dataset, max_length=512):
    references_test = []
    
    em = load("exact_match")
    f1 = load("f1")
    bertscore = load("bertscore")
    
    qa_pipeline = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        device=0 if torch.cuda.is_available() else -1
    )
    
    predictions = []
    
    for example in test_dataset:
        context = " ".join(example["topk_documents"])
        input_text = f"question: {example['question']} context: {context}"
        
        answer = qa_pipeline(
            input_text,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )[0]['generated_text']
        
        predictions.append(normalize_text(answer))
        references_test.append(normalize_text(example["answer"]))
    
    results = {
        "exact_match": em.compute(predictions=predictions, references=references_test, ignore_case=True)["exact_match"],
        #"f1": f1.compute(predictions=predictions, references=references_test)["f1"],
        "bertscore_f1": np.mean(
            bertscore.compute(
                predictions=predictions,
                references=references_test,
                lang="en",
                model_type="bert-base-uncased"
            )["f1"]
        )
    }
    
    return results

#metrics = evaluate_model(model, tokenizer, eval_data)
    
#model = T5ForConditionalGeneration.from_pretrained("your_model_path")
#tokenizer = T5Tokenizer.from_pretrained("your_model_path")
    
#eval_data = Dataset.from_dict({
#   "question": ["Where was Marie Curie born?"],
#   "topk_documents": [["Marie Curie was born in Warsaw, Poland."]], 
#   "answer": ["Warsaw, Poland"]
#  })

In [ ]:
#t = pd_to_hf_ds(topk_df_test, tokenizer)

base_metrics = evaluate_model(model, tokenizer, topk_ds_test)

print("\nMetrics:")
for k, v in base_metrics.items():
    print(f"{k}: {v:.4f}")

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors


### LoRA FT

In [ ]:
def preprocess_function(examples, tokenizer, max_source_length=512, max_target_length=128):
    inputs = []
    for question, documents in zip(examples["question"], examples["topk_documents"]):
        context = " ".join(documents)
        inputs.append(f"question: {question} context: {context}")
    
    model_inputs = tokenizer(
        inputs, 
        max_length=max_source_length, 
        truncation=True, 
        padding="max_length"
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["answer"], 
            max_length=max_target_length, 
            truncation=True, 
            padding="max_length"
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
def create_lora_model(model):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        inference_mode=False,
        r=8, 
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q", "v"]
    )
    
    return get_peft_model(model, lora_config)

In [ ]:
def pd_to_hf_ds(dataset, tokenizer):
    hf_ds = Dataset.from_pandas(dataset)
    tokenized_dataset = hf_ds.map(
        lambda x: preprocess_function(x, tokenizer), batched=True
    )
    return tokenized_dataset

In [ ]:
model_name = "t5-small" # the model name
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model = create_lora_model(model)
model.print_trainable_parameters()

topk_hf_train = pd_to_hf_ds(topk_df_train, tokenizer)
topk_hf_val = pd_to_hf_ds(topk_df_val, tokenizer)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/train",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=3e-4,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    gradient_accumulation_steps=4,
    report_to="tensorboard",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=topk_hf_train,
    eval_dataset=topk_hf_val,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

model.save_pretrained("t5-lora-qa")
tokenizer.save_pretrained("t5-lora-qa")

In [ ]:
ft_metrics = evaluate_model(model, tokenizer, topk_df_test)

print("\nMetrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")